# Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

# Params

In [ ]:
MODEL_DIR = "models"
LOG_ROOT = "logs"
RUN_ID = ""
MODELS = ["drone", "satellite"]
RUNTIMES = ["onnxruntime", "torch"]
THREADS = 4
N_CLASSES = 6

# Data

In [ ]:
samples = {name: np.load(Path(MODEL_DIR) / f"sample-{name}.npz")["x"] for name in MODELS}
{name: x.shape for name, x in samples.items()}

# Model

In [ ]:
def onnx_runner(name: str, threads: int):
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    session = ort.InferenceSession(str(Path(MODEL_DIR) / f"lraspp-{name}.onnx"), options,
                                   providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]

In [ ]:
def torch_runner(name: str, threads: int):
    torch.set_num_threads(threads)
    net = lraspp_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=N_CLASSES)
    state = torch.load(Path(MODEL_DIR) / f"lraspp-{name}.pt", map_location="cpu")
    net.load_state_dict({key.removeprefix("net."): value for key, value in state.items()})
    net.eval()

    @torch.no_grad()
    def run(x: np.ndarray) -> np.ndarray:
        return net(torch.from_numpy(x))["out"].numpy()
    return run

In [ ]:
RUNNER_FACTORIES = {"onnxruntime": onnx_runner, "torch": torch_runner}

# Run

In [ ]:
log_dir = Path(LOG_ROOT) / (RUN_ID or datetime.now().strftime("%Y%m%d-%H%M%S"))
log_dir.mkdir(parents=True, exist_ok=True)
log_dir

In [ ]:
logits = {f"{runtime}-{name}": RUNNER_FACTORIES[runtime](name, THREADS)(samples[name])
          for runtime in RUNTIMES for name in MODELS}
{key: value.shape for key, value in logits.items()}

In [ ]:
np.savez(log_dir / "pi-logits.npz", **logits)